[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aejepsen/ebook-llm-on-premise/blob/main/notebooks/cap12_export_gguf.ipynb)

# Capítulo 12 — Export GGUF e Deploy no Ollama

> **Baseado no projeto [AI-Orchestrator](https://github.com/aejepsen/AI-Orchestrator) de Allan Eric Jepsen**

Neste notebook, vamos:
1. Fazer merge dos adapters LoRA com o modelo base
2. Exportar para formato GGUF (compatível com llama.cpp / Ollama)
3. Quantizar para Q4_K_M (~5.4 GB)
4. Criar o Modelfile do Ollama
5. Testar o modelo final

**Runtime: A100 40GB** (merge requer ~19 GB de RAM para o modelo 9B em 16-bit).

**Pré-requisito**: Adapter LoRA treinado no Google Drive (capítulo anterior).

## 1. Instalação

Mesma instalação pinada do capítulo de treino. Após rodar, faça **Ambiente de execução -> Reiniciar sessão**.

In [ ]:
# Instalação pinada — mesma do capítulo de treino
# APÓS RODAR: Ambiente de execução -> Reiniciar sessão
!pip uninstall -y -q torchaudio 2>/dev/null
!pip install -q --no-cache-dir "unsloth" "unsloth_zoo"
!pip install -q "transformers>=5.2.0,<=5.5.0" "trl>=0.18.2,<=0.24.0" "datasets>=3.4.1,<4.4.0" "accelerate>=1.2" "peft>=0.16" sentencepiece "protobuf<7"
print("INSTALADO — agora: Ambiente de execução -> Reiniciar sessão, depois rode célula 1b")

## 1b. Verificação pós-restart

In [ ]:
import unsloth  # PRIMEIRO
import transformers
print('transformers:', transformers.__version__)
assert transformers.__version__.split('.')[0] == '5', 'transformers errado'
print('OK')

## 2. Montar Drive e carregar modelo com adapter LoRA

Carregamos o modelo base + adapter LoRA treinado. O Unsloth faz o merge automaticamente no export.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
from unsloth import FastLanguageModel

MAX_SEQ = 4096
MODEL_NAME = "unsloth/Qwen3.5-9B"

# Carregar modelo base + adapter LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ,
    dtype           = torch.bfloat16,
    load_in_4bit    = False,
    load_in_16bit   = True,
    full_finetuning = False,
)

# Aplicar LoRA (mesmo config do treino)
model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    lora_alpha     = 32,
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
    max_seq_length = MAX_SEQ,
)

print(f"Modelo carregado: {MODEL_NAME}")
model.print_trainable_parameters()

## 3. Limpar cache HuggingFace

O merge + conversão GGUF precisa de ~25 GB de espaço em disco. O cache do HuggingFace ocupa ~19 GB com o modelo base (que já está carregado na VRAM). Removemos o cache para liberar espaço.

**Essa etapa é obrigatória** — sem ela, o disco do Colab enche durante a conversão.

In [ ]:
import shutil, os

# Remover cache HF (modelo base já carregado em VRAM)
hf_cache = '/root/.cache/huggingface/hub'
if os.path.exists(hf_cache):
    sz = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, fn in os.walk(hf_cache)
        for f in fn
    )
    shutil.rmtree(hf_cache, ignore_errors=True)
    print(f'Cache HF removido: {sz/1024**3:.1f} GB liberados')
else:
    print('Cache HF já limpo')

# Verificar espaço disponível
os.system('df -h /content | tail -1')

## 4. Merge LoRA + Base -> Modelo 16-bit

O merge combina os pesos do adapter LoRA com o modelo base, resultando em um modelo completo. Essa etapa é necessária antes da conversão GGUF.

$$W_{merged} = W_{base} + \frac{\alpha}{r} \cdot B \times A$$

In [ ]:
# Merge dos adapters LoRA no modelo base em 16-bit
# Isso cria uma cópia completa do modelo com os pesos atualizados
print("Fazendo merge LoRA + base (pode levar alguns minutos)...")
model.save_pretrained_merged(
    '/content/merged',
    tokenizer,
    save_method='merged_16bit',  # merge completo em bf16
)
print("Merge concluído!")

# Verificar tamanho
merged_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk('/content/merged')
    for f in fn
)
print(f"Modelo merged: {merged_size / 1024**3:.1f} GB")

## 5. Conversão GGUF Q4_K_M

GGUF é o formato nativo do llama.cpp (e Ollama). A quantização Q4_K_M oferece o melhor balanço entre qualidade e tamanho para modelos 9B:

| Quantização | Tamanho (9B) | Qualidade | Uso |
|------------|-------------|-----------|-----|
| F16 | ~18 GB | Máxima | Referência |
| Q8_0 | ~9 GB | Excelente | Quando VRAM permite |
| **Q4_K_M** | **~5.4 GB** | **Muito boa** | **Produção (AI-Orchestrator)** |
| Q4_K_S | ~5.1 GB | Boa | Economia máxima |

**Gotcha Unsloth**: o GGUF é gerado em `gguf_gguf/` (Unsloth adiciona `_gguf` ao path informado). Precisamos buscar o arquivo correto.

In [ ]:
import glob, os, shutil

DST = '/content/drive/MyDrive/ai-orchestrator-lora'
os.makedirs(DST, exist_ok=True)

# Conversão GGUF Q4_K_M
# Unsloth compila llama.cpp na primeira chamada (leva ~2-3 min)
print("Convertendo para GGUF Q4_K_M (pode levar 10-15 min)...")
model.save_pretrained_gguf(
    '/content/gguf',
    tokenizer,
    quantization_method='q4_k_m',  # melhor balanço qualidade/tamanho
)

# Localizar o GGUF gerado
# GOTCHA: Unsloth gera em /content/gguf_gguf/ (adiciona _gguf ao path)
ggufs = sorted(
    glob.glob('/content/gguf_gguf/**/*.gguf', recursive=True) +
    glob.glob('/content/gguf/**/*.gguf', recursive=True),
    key=os.path.getsize,
    reverse=True,
)
assert ggufs, 'Nenhum .gguf gerado — verifique /content/gguf e /content/gguf_gguf'

# Pegar o Q4_K_M
src = [g for g in ggufs if 'Q4_K_M' in g]
if src:
    src = src[0]
else:
    src = ggufs[0]  # fallback: maior arquivo

gguf_name = 'qwen3.5-9b-orch.Q4_K_M.gguf'
shutil.copy2(src, f'{DST}/{gguf_name}')

size_gb = os.path.getsize(src) / 1024**3
print(f'\nGGUF copiado para Drive: {DST}/{gguf_name}')
print(f'Tamanho: {size_gb:.1f} GB')

## 6. Criar Modelfile do Ollama

O Modelfile define como o Ollama carrega e executa o modelo. Inclui:
- Path para o GGUF
- Template de chat (ChatML com suporte a tools)
- Parâmetros de inferência (temperature, top_p, etc.)
- Tokens de parada

O template abaixo suporta tool-calling no formato que o AI-Orchestrator espera.

In [ ]:
# Template Go do Ollama para Qwen3.5 com suporte a tools
# Esse template é o mesmo usado no AI-Orchestrator em produção
GO_TEMPLATE = '''{{- if .Messages }}
{{- if or .System .Tools }}<|im_start|>system
{{ .System }}
{{- if .Tools }}

# Tools

You may call one or more functions to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{{- range .Tools }}
{"type": "function", "function": {{ .Function }}}
{{- end }}
</tools>

For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>
{{- end }}<|im_end|>
{{ end }}
{{- range $i, $_ := .Messages }}
{{- $last := eq (len (slice $.Messages $i)) 1 -}}
{{- if eq .Role "user" }}<|im_start|>user
{{ .Content }}<|im_end|>
{{ else if eq .Role "assistant" }}<|im_start|>assistant
{{ if .Content }}{{ .Content }}
{{- end }}
{{- if .ToolCalls }}<tool_call>
{{ range .ToolCalls }}{"name": "{{ .Function.Name }}", "arguments": {{ .Function.Arguments }}}
{{ end }}</tool_call>
{{- end }}{{ if not $last }}<|im_end|>
{{ end }}
{{- else if eq .Role "tool" }}<|im_start|>user
<tool_response>
{{ .Content }}
</tool_response><|im_end|>
{{ end }}
{{- if and (ne .Role "assistant") $last }}<|im_start|>assistant
{{ end }}
{{- end }}
{{- else }}
{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}{{ if .Prompt }}<|im_start|>user
{{ .Prompt }}<|im_end|>
{{ end }}<|im_start|>assistant
{{ end }}{{ .Response }}{{ if .Response }}<|im_end|>{{ end }}'''

# Gerar Modelfile
gguf_name = 'qwen3.5-9b-orch.Q4_K_M.gguf'

modelfile = f'''FROM ./{gguf_name}

TEMPLATE """{GO_TEMPLATE}"""

PARAMETER stop "<|im_start|>"
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.7
PARAMETER top_p 0.8
PARAMETER top_k 20
PARAMETER repeat_penalty 1.0
PARAMETER num_ctx 8192
'''

DST = '/content/drive/MyDrive/ai-orchestrator-lora'
modelfile_path = f'{DST}/Modelfile'
with open(modelfile_path, 'w') as fh:
    fh.write(modelfile)

print(f'Modelfile escrito em: {modelfile_path}')
print(f'\nConte\u00fado (primeiras 20 linhas):')
for i, line in enumerate(modelfile.split('\n')[:20]):
    print(f'  {line}')

## 7. Deploy local — passo a passo

Após baixar o `.gguf` e o `Modelfile` do Google Drive para a máquina local:

### 7.1 Criar o modelo no Ollama
```bash
# Navegar até o diretório com os arquivos
cd ~/ai-orchestrator-lora/

# Criar modelo (leva ~1 min para indexar)
ollama create qwen3.5-9b-orch -f Modelfile
```

### 7.2 Testar interativamente
```bash
ollama run qwen3.5-9b-orch
```

### 7.3 Rodar gates de avaliação
```bash
MODEL=qwen3.5-9b-orch python evals/eval_routing.py     # gate: >= 90%
MODEL=qwen3.5-9b-orch python evals/eval_injection.py   # gate: 0 leaks
MODEL=qwen3.5-9b-orch python evals/eval_domains.py     # gate: >= 80%/domínio
```

### 7.4 Promover para produção (se aprovado)
```bash
# Atualizar .env
echo 'MODEL=qwen3.5-9b-orch' >> .env

# Restart do gateway
docker-compose restart gateway
```

## 8. Verificar integridade do GGUF

Antes de baixar, verificamos que o GGUF é válido lendo o header.

In [ ]:
import struct
import os

def verify_gguf(path: str) -> dict:
    """Verifica integridade básica de um arquivo GGUF lendo o header."""
    info = {'path': path, 'valid': False}
    
    if not os.path.exists(path):
        info['error'] = 'Arquivo não encontrado'
        return info
    
    info['size_gb'] = os.path.getsize(path) / 1024**3
    
    with open(path, 'rb') as f:
        # Magic number: "GGUF" (4 bytes)
        magic = f.read(4)
        if magic != b'GGUF':
            info['error'] = f'Magic number inválido: {magic}'
            return info
        
        # Versão (uint32 LE)
        version = struct.unpack('<I', f.read(4))[0]
        info['version'] = version
        
        # Número de tensores (uint64 LE)
        n_tensors = struct.unpack('<Q', f.read(8))[0]
        info['n_tensors'] = n_tensors
        
        # Número de metadados (uint64 LE)
        n_kv = struct.unpack('<Q', f.read(8))[0]
        info['n_metadata'] = n_kv
        
        info['valid'] = True
    
    return info

# Verificar GGUF no Drive
DST = '/content/drive/MyDrive/ai-orchestrator-lora'
gguf_path = f'{DST}/qwen3.5-9b-orch.Q4_K_M.gguf'

if os.path.exists(gguf_path):
    info = verify_gguf(gguf_path)
    print(f"GGUF: {info['path']}")
    print(f"  Válido: {info['valid']}")
    print(f"  Tamanho: {info.get('size_gb', 0):.1f} GB")
    print(f"  Versão GGUF: {info.get('version', 'N/A')}")
    print(f"  Tensores: {info.get('n_tensors', 'N/A'):,}")
    print(f"  Metadados: {info.get('n_metadata', 'N/A'):,}")
else:
    print(f"GGUF não encontrado em {gguf_path}")
    print("Execute as células de merge e conversão primeiro.")

## 9. Teste rápido do modelo merged

Antes de baixar o GGUF, podemos testar o modelo merged (ainda na VRAM do Colab) para verificar que o merge não corrompeu os pesos.

In [ ]:
# Teste rápido com o modelo merged (ainda na VRAM)
FastLanguageModel.for_inference(model)

# Prompt de teste no formato do AI-Orchestrator
test_messages = [
    {"role": "system", "content": "Voc\u00ea \u00e9 o classificador de inten\u00e7\u00f5es do orquestrador corporativo. Analise a pergunta e retorne um JSON com os dom\u00ednios necess\u00e1rios."},
    {"role": "user", "content": "Pergunta: Qual o saldo atual do cliente Jo\u00e3o Silva?"},
]

# Aplicar chat template e tokenizar
text = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors='pt').to(model.device)

# Gerar resposta
import torch
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.1,
        do_sample=True,
    )

# Decodificar apenas os tokens gerados
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print("Pergunta: Qual o saldo atual do cliente Jo\u00e3o Silva?")
print(f"Resposta: {response}")
print("\nSe a resposta cont\u00e9m JSON com 'domains', o merge est\u00e1 OK.")

## 10. Resumo do pipeline completo

```
Dataset SFT (3.050 exemplos)
    │
    ▼
Treino LoRA (Unsloth, A100, 148 min)
    │  r=16, alpha=32, 2 epochs
    ▼
Adapter LoRA (~100 MB)
    │
    ▼
Merge LoRA + Base -> Modelo 16-bit (~18 GB)
    │
    ▼
Quantização GGUF Q4_K_M (~5.4 GB)
    │
    ▼
Ollama create (Modelfile + GGUF)
    │
    ▼
Gates de avaliação (routing/injection/domains)
    │
    ▼
Produção (.env MODEL=qwen3.5-9b-orch)
```

### Gotchas documentadas (AI-Orchestrator):

1. **Unsloth gera GGUF em `gguf_gguf/`** (adiciona `_gguf` ao path)
2. **Limpar cache HF antes do export** (disco do Colab enche)
3. **Ollama 0.24 não suporta arquitetura híbrida DeltaNet** — atualize para >= 0.30
4. **`keep_alive` deve ser int** (Ollama 0.30 rejeita string `"-1"`)
5. **`think=true` não funciona em Qwen3.5** (Small series não suporta)